In [ ]:
using DelimitedFiles    #Para generar y leer archivos csv
using CairoMakie        #Para generar las gráficas
using LaTeXStrings      #Para darle formato con LaTeX a los títulos y ejes de las gráficas
using ConcaveHull       #Para realizar un Concave Hull de los histogramas del g(R)
using StatsBase         #Para emplear algunas funciones básicas de estadística, como el promedio
using LsqFit;           #Paquetería para realizar un ajuste por mínimos cuadrados
using Statistics;       #Paquetería para realizar cálculos de estadística básica
using FFTW;             #Paquetería para realizar análisis de Fourier

#Función que da el aproximante estadístico de los sistemas cuasiperiódicos tras considerar el factor de normalización de Torquato
AproxLambda(x) = (2π/(1 - cos(2π/x)));

#Función que "suaviza" una gráfica reduciendo el número de puntos por su promedio (media móvil)
function promedio_m(A, i, m, Factor)
    Valores = [Factor*A[j][2] for j in (i-m):(i+m)]
    return mean(Valores)
end

In [ ]:
#Diccionario con los valores de la densidad numérica de los sistemas cuasiperiódicos en 2D
Rho_Dict = Dict(
                5  => 1.2328979808609704,
                7  => 1.2517957581185175,
                9  => 1.260284085456272,
                11 => 1.2645739922427748,
                13 => 1.2670366156186388,
                15 => 1.2685820147323164,
                17 => 1.2696141740269873,
                19 => 1.2703373895977548,
                21 => 1.2708642307351703,
                23 => 1.2712590235307708,
                25 => 1.271563821555834,
                27 => 1.271802352106815,
                29 => 1.2719940413420914,
                31 => 1.2721488757491926
               );

### Cargar los datos de la g(R)

In [ ]:
NSides = 19;            #Simetría del arreglo cuasiperiódico a analizar
Radius = 500;           #Radio de las vecindades circulares
MuestraNotebook = 500;  #Número de iteraciones realizadas por cada notebook
NumNotebooks = 20;      #Número total de notebooks que se ejecutaron
RandSteps = 0;          #Número de iteraciones del algoritmo de randomización

#Ruta de acceso a la carpeta con los datos del Histograma de g(R) en decorado en vértices
Path = "Quasiperiodic-Tiles/Global Structural Studies/Data/Fig4_gR/";

#Lectura de datos de los histogramas para la construcción de la g(R)
PesosHist = vec(readdlm(Path * "Histograma_Pesos_N$(NSides)_Muestra$(MuestraNotebook)_R$(Radius)_S0P001_1.csv"));
for i in 2:NumNotebooks
    PesosHist .+= vec(readdlm(Path * "Histograma_Pesos_N$(NSides)_Muestra$(MuestraNotebook)_R$(Radius)_S0P001_$(i).csv"));
end

Rho = 1.2703373895977548;                                   #Densidad promedio de sitios del decorado en Vértices;
Δ = 0.001;                                                  #Tamaño del crecimiento en el radio de la ventana circular
Rango = 0.0:Δ:Radius;                                       #El intervalo de radios R empleados para generar los datos
PesosHist = PesosHist ./ (NumNotebooks * MuestraNotebook);  #Calculamos el promedio de los datos que formaron al arreglo PesosHist

#NOTA: La variable "Factor" se compone de dos términos. El primero (1/Δ) es básicamente el inverso del 'Step' empleado en la variación del R para generar los datos originales de los que se obtuvo la altura del histograma
#      contenido en los archivos. El segundo término (1/100) se emplea para "contrarrestar" un factor de '100' que se introduce más adelante a la altura de los histogramas 'PesosHist', este factor de '100' se usa para optimizar 
#      el cálculo de las envolventes con la función ConcaveHull y no se requiere para ninguna otra función.
Factor = (1/Δ)/100;

#NOTA: El Factor de Normalización que introduce Torquato en su artículo sobre Hiperuniformidad (aparece en el Suplemento) toma la forma 'Factor_Normalizacion = 1/(2*sqrt(π*Rho))'. El inverso de este Factor es el que se requiere 
#      al multiplicar las distancias, motivo por el que aparece dicha expresión en la normalización del Rango.
#NOTA 2: La normalización de la frecuencia se realiza con los valores de R y ΔR antes del factor introducido por Torquato porque es una normalización al histograma que se obtuvo antes del factor de Torquato.
Rango = [(Rango[i+1] + Rango[i])/2 for i in 1:(length(Rango) - 1)]; #Arreglo con los valores centrales de cada bin del histograma
RangoNorm = 2*sqrt(π*Rho)*Rango; #Re-escalamos las longitudes con el factor de Torquato
Frecuencia = 100*(PesosHist .+ 1e-6)./(2*π*Rho*Rango); #Frecuencias de los histogramas normalizados

### **Visualización de los datos en forma de histograma interactivo

In [ ]:
using Plots
plotly()

Lambda = AproxLambda(NSides);                   #Longitud del aproximante estadístico
κ = (NSides/4)*(2*sqrt(π*Rho));                 #Longitud de escala kappa
SI = 1002;                                      #Índice que marca el inicio del análisis de los datos
EI = findfirst(x -> x >= 6*Lambda, RangoNorm);  #Índice que marca el final del análisis de los datos
gR_Min = minimum(Factor .* Frecuencia[SI:EI]);  #Valor mínimo que alcanza la g(R)
gR_Max = maximum(Factor .* Frecuencia[SI:EI]);  #Valor máximo que alcanza la g(R)

#Gráfica del g(R)
Plots.plot(
           RangoNorm[SI:EI], Factor .* Frecuencia[SI:EI],
           label = "",
           title = L"N = %$(NSides)",
           xlabel = L"R",
           ylabel = L"g(R)",
           yscale = :log10
          )

#Línea horizontal para el límite teórico de la g(R)
Plots.plot!(
            [RangoNorm[SI], RangoNorm[EI]], [1, 1],
            label = L"y = 1",
            linestyle = :dash,
            color = :black
           )

#Múltiplos de 2*Kappa sobre la g(R)
for i in 2:2:2
      Plots.plot!(
                  [i*κ, i*κ], [gR_Min, gR_Max],
                  label = L"\kappa \approx N",
                  linestyle = :dash,
                  color = :green
                 )
end

#Múltiplos de 2*Lambda sobre la g(R)
for i in 2:2:6
      Plots.plot!(
                  [i*Lambda, i*Lambda], [gR_Min, gR_Max],
                  linestyle = :dash,
                  color = :red
                 )
end

Plots.plot!(
            legend = false,
            #xlimit = [4, 30]
           )

### **Obtenemos sólo los puntos que caen por arriba de la horizontal a altura 1 (Opcional)

In [ ]:
Superior_gR = [];   #Valores de g(R) de los sitios que caen por arriba de la línea con altura 1
Superior_R = [];    #Valores de R de los sitios que caen por arriba de la línea con altura 1
Altura = 1;         #Altura de la línea límite a la que en teoría tienden los histogramas de la g(R)

R_Acotado = RangoNorm[SI:EI];
gR_Acotado = Factor .* Frecuencia[SI:EI];

#Separación de los valores que caen por arriba de la horizontal y = 1
for Sitio_Indice in 1:length(R_Acotado)
    if gR_Acotado[Sitio_Indice] >= Altura
        push!(Superior_gR, gR_Acotado[Sitio_Indice]);
        push!(Superior_R, R_Acotado[Sitio_Indice]);
    end
end

gR_Sup_Min = minimum(Superior_gR);  #Valor mínimo que alcanza la g(R) en sus valores superiores a 1
gR_Sup_Max = maximum(Superior_gR);  #Valor máximo que alcanza la g(R) en sus valores superiores a 1

#Gráfica g(R) sólo la parte arriba de y = 1
Plots.plot(
           Superior_R, Superior_gR .- 1,
           label = "",
           title = L"N = %$(NSides)",
           xlabel = L"R",
           ylabel = L"g(R)",
           yscale = :log10,
           xscale = :log10
          )

#Múltiplos de 2*Kappa sobre la g(R)
for i in 2:2:12
    Plots.plot!(
                [i*κ, i*κ], [gR_Sup_Min - 1, gR_Sup_Max],
                label = L"\kappa \approx N",
                linestyle = :dash,
                color = :green
               )
end

#Múltiplos de 2*Lambda sobre la g(R)
for i in 2:2:6
      Plots.plot!(
                  [i*Lambda, i*Lambda], [gR_Sup_Min - 1, gR_Sup_Max],
                  linestyle = :dash,
                  color = :red
                 )
end

Plots.plot!(
            legend = false,
            #size = (1800, 800), 
           )

### **Calculamos la Concave Hull del histograma

In [ ]:
Puntos = [[RangoNorm[i], Frecuencia[i]] for i in SI:EI];                    #Coordenadas de los puntos que conforman la gráfica
PCH = 282;                                                                  #Número de vecinos alrededor de cada punto para realizar el Concave Hull
Hull = concave_hull(Puntos, PCH)                                            #Se calcula el Concave Hull para las gráficas generadas

### **Visualizamos la Concave Hull sobre el histograma

In [ ]:
Plots.plot(
           RangoNorm[SI:EI], Factor .* Frecuencia[SI:EI],
           label = "",
           title = L"N = %$(NSides)",
           xlabel = L"R",
           ylabel = L"g(R)",
           yscale = :log10,
           xscale = :log10
          )

#Gráfica del g(R) con el ConcaveHull encima
Plots.plot!(
            [i[1] for i in Hull.vertices], [Factor .* i[2] for i in Hull.vertices],
            grid = false,
            lw = 2,
            legend = false
           )

### **Separación del Concave Hull en envolvente superior e inferior

In [ ]:
VerticesSuperior = [];  #Coordenadas de los sitios que caen por arriba de la línea con altura 1
VerticesInferior = [];  #Coordenadas de los sitios que caen por debajo de la línea con altura 1
altura = 1;             #Altura de la línea límite a la que en teoría tienden los histogramas de la g(R)

#Iteramos sobre todos los sitios del Concave Hull para separar los que caen arriba de la línea horizontal de los que caen debajo
for i in Hull.vertices
    if Factor*i[2] >= altura
        push!(VerticesSuperior, i)
    else
        push!(VerticesInferior, i)
    end
end

#Acomodamos los vértices de los Concave Hull de izquierda a derecha
sort!(VerticesSuperior, by=x->x[1]);
sort!(VerticesInferior, by=x->x[1]);

### **Visualización de la envolvente superior del Concave Hull

In [ ]:
m = 0; #Valor para la media móvil
#PARTE SUPERIOR DE LA ENVOLVENTE
VSProm = [[VerticesSuperior[i][1], promedio_m(VerticesSuperior, i, m, Factor)] for i in (1+m):(length(VerticesSuperior)-m)]; #Arreglo con los sitios de la media móvil de la envolvente superior

#Gráfica de la g(R)
Plots.plot(
           RangoNorm[SI:EI], Factor .* Frecuencia[SI:EI],
           ms = 1,
           label = "",
           markerstrokewidth = 0.0,
           color = :gray,
           alpha = 0.75
          )

#Gráfica de la envolvente superior de la Concave Hull
Plots.plot!(
            [VSProm[i][1] for i in 1:length(VSProm)], [VSProm[i][2] for i in 1:length(VSProm)],
            markersize = 1,
            legend=false,
            color = 3,
            lw = 2
           )

#Múltiplos de 2*Kappa sobre la g(R)
for i in 2:2:12
    Plots.plot!(
                [i*κ, i*κ], [gR_Sup_Min - 1, gR_Sup_Max],
                label = L"\kappa \approx N",
                linestyle = :dash,
                color = :green
               )
end

#Múltiplos de 2*Lambda sobre la g(R)
for i in 2:2:6
      Plots.plot!(
                  [i*Lambda, i*Lambda], [gR_Sup_Min - 1, gR_Sup_Max],
                  linestyle = :dash,
                  color = :red
                 )
end

#Parámetros para estilo de la gráfica para imagen
Plots.plot!(
            scale = :log10
           )

### Erdal's line slope comprobation

In [ ]:
m = 0; #Valor para la media móvil
#PARTE SUPERIOR DE LA ENVOLVENTE
VSProm = [[VerticesSuperior[i][1], promedio_m(VerticesSuperior, i, m, Factor)] for i in (1+m):(length(VerticesSuperior)-m)]; #Arreglo con los sitios de la media móvil de la envolvente superior

#Gráfica de la (g(R) - 1) con la envolvente superior de la Concave Hull
Plots.plot(
           [VSProm[i][1] for i in 1:length(VSProm)], [VSProm[i][2] - 1 for i in 1:length(VSProm)],
           label = "",
           markersize = 1,
           color = 3,
           lw = 2
          )
      

Lambda = AproxLambda(NSides); #Longitud del aproximante estadístico
R1 = 4:2*Lambda
Plots.plot!(R1, 3000 .* (R1 .^ (-1.2)),
      markersize = 1,
      color = :red,
      lw = 2)

R2 = 2*Lambda:4*Lambda
Plots.plot!(R2, 6000 .* (R2 .^ (-1.2)),
      markersize = 1,
      color = :red,
      lw = 2)

R3 = 4*Lambda:6*Lambda
Plots.plot!(R3, 8000 .* (R3 .^ (-1.2)),
      markersize = 1,
      color = :purple,
      lw = 2)

#Múltiplos de 2*Kappa sobre la g(R)
for i in 2:2:12
    Plots.plot!(
                [i*κ, i*κ], [0.15, gR_Sup_Max],
                label = L"\kappa \approx N",
                linestyle = :dash,
                color = :green
               )
end

#Múltiplos de 2*Lambda sobre la g(R)
for i in 2:2:6
      Plots.plot!(
                  [i*Lambda, i*Lambda], [0.15, gR_Sup_Max],
                  linestyle = :dash,
                  color = :red
                 )
end

Plots.plot!(
            scale = :log10,
            legend = false
           )

### Exportación de la gráfica con Cairo Makie

Gráfica "Maestra" de las $\sigma^2$ re-escaladas

In [ ]:
#Modelo 2: Promedio
#Starting Point = λ
Dict_Lambda_Infty_STD = Dict(
                             5  => [0.14842677130532758, 0.02693208062674656],  #Actualizado
                             7  => [0.20599942249634387, 0.02889749910608079],  #Actualizado
                             9  => [0.20511073236241031, 0.025078052118710375], #Actualizado
                             11 => [0.28812778652572674, 0.05343895152132179],  #Actualizado
                             13 => [0.3478715456767856, 0.07341374061357663],   #Actualizado
                             15 => [0.34865315655359685, 0.08051135345795574],  #Actualizado
                             17 => [0.4593732477904627, 0.11864843159480597],   #Actualizado
                             19 => [0.5337295342446887, 0.145608944161264],     #Actualizado
                             21 => [0.5535341355465558, 0.15582047558893325],   #Actualizado
                             23 => [0.6813762592148858, 0.1963541228475375],    #Actualizado
                             25 => [0.7593901994594552, 0.22370865721408364],   #Actualizado
                             27 => [0.7848784960127629, 0.23360456762197016],   #Actualizado
                             29 => [0.9205104301894491, 0.27222890717914594],   #Actualizado
                             31 => [1.0268510348123494, 0.3089886592548087]     #Actualizado
                            );

# 1. Definimos el modelo: y = a + b * x^c
# p[1] = a, p[2] = b, p[3] = c
# Usamos el operador punto (.) para operaciones vectorizadas
@. model(x, p) = p[1] + p[2] * x^p[3]

# 2. Datos a ajustar
xdata = [i for i in 5:2:31]
ydata = [Dict_Lambda_Infty_STD[i][1] for i in 5:2:31]

# 3. Valores iniciales (Crucial para modelos no lineales)
p0 = [1.0, 1.0, 1.0]

# 4. Ejecutar el ajuste
fit = curve_fit(model, xdata, ydata, p0)

# 5. Extraer los resultados
p_opt = fit.param
println("Parámetros ajustados: a = $(p_opt[1]), b = $(p_opt[2]), c = $(p_opt[3])")

In [ ]:
DataPath = "Quasiperiodic-Tiles/Global Structural Studies/Data/Fig4_SigmaSquare_2D/"; #Ruta de acceso a los datos para analizar

#Datos de las vecindades generadas al obtener los datos
Radio = 500;            #Radio de las vecindades circulares
Pasos = 1e5;            #Número de pasos a dar desde R = 0 hasta R = Radio para variar el radio de la ventana circular de N(R)
ΔStep = Radio/Pasos;    #Salto en los valores de R

# --- Definimos el lienzo general y el eje donde se graficará la primera imagen en el lienzo completo ---
Fig = Figure(size = (1800, 600), figure_padding = (10, 40, 10, 5));             #Para el size (Alto, Ancho); #Para el padding (izq, der, abajo, arriba)
Master_Sigma2_2D = Axis(
                        Fig[1,1],                                               #Posición relativa de la gráfica dentro del lienzo completo de la figura
                        xlabel = L"R / λ_{N}",                                  #Etiqueta para el eje horizontal
                        ylabel = L"\sigma^{2}(R) / (R \Lambda_{\infty}(N))",    #Etiqueta para el eje vertical
                        yticks = (
                                  [0, 1.5],                                     #Posiciones donde colocar los ticks en el eje Y
                                  [rich("0.0"), rich("1.5")]                    #Etiquetas de los ticks en el eje Y
                                 ),
                        xticklabelsize = 45, yticklabelsize = 45,
                        xlabelsize = 45, ylabelsize = 45,
                        limits = ((0, 3), nothing),                             #Límites de visualización de la gráfica en formato (xlimit, ylimit)
                        rightspinevisible = false,                              #Oculta la línea vertical derecha
                        topspinevisible = false,                                #Oculta la línea horizontal superior
                        xgridvisible = false,                                   #Mallado en los ejes horizontales
                        ygridvisible = false,                                   #Mallado en los ejes verticales
                       );

#Definimos el gradiente de colores para la gráfica
mi_paleta = cgrad(:inferno, 8, categorical = true)

Indice_Grafica = 1; 
for NSides in [11, 13, 17, 19, 23, 29, 31]
    #Datos de la Sigma^2
    σ2 = vec(readdlm(DataPath * "N$(NSides)/NR_V/Torquato_NR_N$(NSides)_Alfa0P0_R$(Radio)_Step1e5_SigmaCuadrada.csv"));

    Rho = Rho_Dict[NSides]; #Densidad numérica de sitios en la retícula cuasiperiódica
    FN = 2*sqrt(π*Rho); #Factor de normalización para mantener los resultados independientes de la densidad de puntos

    #Generamos el intervalo con los valores de la R asociados a los datos de sigma cuadrada (Incluye factor de 2*sqrt(π*Rho)
    #necesario para mantener densidad de puntos constantes en decorado, independientemente de la simetría rotacional)
    R = ΔStep:ΔStep:Radio;
    R = FN .* R;

    λ = AproxLambda(NSides); #Longitud de escala de nuestro sistema
    Λ = p_opt[1] + p_opt[2] * (NSides ^ p_opt[3])

    #Graficamos las fluctuaciones de N(R) al cuadrado, sobre R, como función de R
    lines!(Master_Sigma2_2D, R ./ λ, σ2 ./ (R .* Λ),
           color = (mi_paleta[Indice_Grafica], 1.0),
           label = L"N = %$(NSides)")
    Indice_Grafica += 1;
end

Fig

In [ ]:
# --- Definimos el lienzo general y el eje donde se graficará la primera imagen en el lienzo completo ---
gR_Pwr_Law = Axis(
                  Fig[1,2],               #Posición relativa de la gráfica dentro del lienzo completo de la figura
                  yscale = log10,         #Escala logarítmica para el eje vertical
                  xscale = log10,         #Escala logarítmica para el eje horizontal
                  xlabel = L"R",          #Etiqueta para el eje horizontal
                  ylabel = L"g(R) - 1",   #Etiqueta para el eje vertical
                  xticks = (
                            [
                             10, 20, 30, 40, 50, 60, 70, 80, 90, 100,
                             200, 300, 400, 500, 600, 700, 800, 900, 1000     #Posiciones donde colocar los ticks en el eje X
                            ],
                            [
                             rich("10"), "", "", "", "", "", "", "", "", 
                             "", "", "", "", "", "", rich("700"), "", "",     #Etiquetas de los ticks en el eje X
                             ""
                            ]
                           ),
                  yticks = (
                            [
                             0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1,
                             2, 3, 4, 5, 6, 7, 8, 9, 10,
                             20, 30, 40, 50, 60, 70, 80, 90, 100,             #Posiciones donde colocar los ticks en el eje X
                             200, 300, 400, 500, 600, 700, 800, 900, 1000
                            ],
                            [
                             "", "", "", "", "", "", "", "", "", rich("10"),
                             "", "", "", "", "", "", "", "", "",
                             "", "", "", "", "", "", "", "", rich("100"),     #Etiquetas de los ticks en el eje X
                             "", "", "", "", "", "", "", "", ""
                            ]
                           ),
                  xticklabelsize = 45, yticklabelsize = 45,
                  xlabelsize = 45, ylabelsize = 45,
                  limits = ((10, VSProm[end][1] + 10), nothing),  #Límites de visualización de la gráfica en formato (xlimit, ylimit)
                  rightspinevisible = false,                      #Oculta la línea vertical derecha
                  topspinevisible = false,                        #Oculta la línea horizontal superior
                  xgridvisible = false,                           #Mallado en los ejes horizontales
                  ygridvisible = false,                           #Mallado en los ejes verticales
                 );

lines!(
       gR_Pwr_Law, [VSProm[i][1] for i in 1:length(VSProm)], [VSProm[i][2] - 1 for i in 1:length(VSProm)],
       color = :orange
      )

vlines!(
        gR_Pwr_Law, [i*Lambda for i in 2:2:6],
        linestyle = :dash,
        linewidth = 3,
        color = (:black, 0.3)
       )

R1 = 4:2*Lambda
lines!(
       gR_Pwr_Law, R1, 2750 .* (R1 .^ (-1.2)),
       color = :red,
       linestyle = :dash,
       linewidth = 3
      )

R2 = 2*Lambda:4*Lambda
lines!(
       R2, 6000 .* (R2 .^ (-1.2)),
       color = :red,
       linestyle = :dash,
       linewidth = 3
      )

R3 = 4*Lambda:6*Lambda
lines!(
       R3, 8000 .* (R3 .^ (-1.2)),
       color = :red,
       linestyle = :dash,
       linewidth = 3
      )

Fig